[![Abrir en Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/Smiledxd/python_data/blob/main/sesiones/S27_comunicar_resultados.ipynb)

# Sesión 27 · Comunicar resultados

**Módulo 6: Negocio y extras** · ⏱️ Duración estimada: 60 minutos en el notebook, más el cierre del proyecto (P4), la publicación (Post 3) y la presentación final

## 🎯 Objetivos
Al terminar esta sesión podrás:
1. Presentar números como los lee quien decide: en soles, millones y porcentajes legibles.
2. Estructurar una propuesta de negocio empezando por la conclusión.
3. Preparar gráficos y tablas listos para una diapositiva.
4. Citar tus fuentes de forma consistente y dejar un notebook limpio y reproducible.
5. Armar un resumen ejecutivo de una página y preparar tu presentación final.

## 📋 Qué debes saber antes
Funciones (sesión 3), pandas (módulo 3) y gráficos que comunican (sesiones 15 y 16).

## 🧭 Cómo trabajar este notebook
- Ejecuta las celdas **en orden**, de arriba abajo, con **Shift + Enter**.
- En cada ✍️ **Tu turno** escribe tu código debajo de `# Tu código aquí`, con los nombres de variables que se piden.
- Después ejecuta la celda ✅ **Verificar**. Si aparece ❌, lee el motivo, corrige y vuelve a verificar. En los textos, el verificador revisa la forma (largo, números, estructura); el contenido lo decides tú.
- Si te atascas, abre la 💡 **Pista**. Hay dos, de menos a más ayuda.

Esta es la **última sesión del plan**. Después del notebook vienen el 🧱 **cierre del proyecto** (P4 completo, con el dashboard en Power BI), la guía del 📣 **Post 3** y la de la presentación final.

## ⚙️ Setup
Ejecuta la celda siguiente **al empezar** (y otra vez si reinicias el entorno). Genera los datos, aplica el estilo de gráficos y carga las funciones que revisan tus respuestas.

⚠️ **Ejecútala y no la edites.**

In [ ]:
#@title ⚙️ Setup: ejecuta esta celda y no la edites { display-mode: "form" }
# Prepara los datos de la sesión, aplica el estilo de gráficos y carga los verificadores.
import copy
import hashlib
import math
import os
import statistics

import matplotlib as mpl
import matplotlib.colors as mcolors
import matplotlib.pyplot as plt
import numpy as np
import pandas as pd

rng = np.random.default_rng(42)

# ---------- Estilo de los gráficos ----------
# Paleta categórica en orden fijo (validada para daltonismo) y tintas para textos y ejes.
AZUL, NARANJA, AQUA, AMARILLO, MAGENTA, VERDE, VIOLETA, ROJO = (
    "#2a78d6", "#eb6834", "#1baf7a", "#eda100", "#e87ba4", "#008300", "#4a3aa7", "#e34948")
GRIS = "#c3c2b7"          # para lo que no es protagonista
TINTA = "#0b0b0b"         # textos principales
TINTA_2 = "#52514e"       # textos secundarios
FONDO = "#fcfcfb"
plt.rcParams.update({
    "figure.facecolor": FONDO, "axes.facecolor": FONDO, "savefig.facecolor": FONDO,
    "axes.edgecolor": GRIS, "axes.labelcolor": TINTA_2, "text.color": TINTA,
    "xtick.color": "#898781", "ytick.color": "#898781",
    "axes.spines.top": False, "axes.spines.right": False,
    "axes.grid": True, "axes.grid.axis": "y", "grid.color": "#e1e0d9", "grid.linewidth": 0.8, "axes.axisbelow": True,
    "axes.prop_cycle": plt.cycler(color=[AZUL, NARANJA, AQUA, AMARILLO, MAGENTA, VERDE, VIOLETA, ROJO]),
    "axes.titlesize": 13, "axes.titlelocation": "left", "axes.titleweight": "bold",
    "lines.linewidth": 2, "font.size": 11, "figure.dpi": 100,
    "axes.formatter.useoffset": False, "axes.formatter.limits": (-9, 9),   # sin notación científica (1e6)
})



# ---------- Datos de práctica: resultados anuales de una cadena de tiendas ----------
_base = np.array([12.4e6, 4.2e6, 3.8e6, 2.1e6, 0.9e6])
_crec = np.array([0.055, 0.16, 0.025, 0.19, 0.33]) + rng.normal(0, 0.01, 5)
ventas_region = pd.DataFrame({
    "region": ["Lima", "Norte", "Sur", "Centro", "Oriente"],
    "ventas_2023": np.round(_base * rng.normal(1, 0.02, 5), -2),
})
ventas_region["ventas_2024"] = np.round(ventas_region["ventas_2023"] * (1 + _crec), -2)
FUENTES = [
    {"autores": "Instituto Nacional de Estadística e Informática", "anio": 2024, "titulo": "Encuesta Nacional de Hogares",
     "tipo": "Dataset", "editor": "INEI", "url": "https://www.inei.gob.pe"},
    {"autores": "Moro, S., Rita, P., & Cortez, P.", "anio": 2014, "titulo": "Bank Marketing", "tipo": "Dataset",
     "editor": "UCI Machine Learning Repository", "url": "https://doi.org/10.24432/C5K306"},
    {"autores": "Cadena de tiendas (datos simulados del curso)", "anio": 2025, "titulo": "Ventas por región 2023-2024",
     "tipo": "Datos simulados", "editor": "Notebook S27", "url": None},
]

_D = copy.deepcopy({"ventas_region": ventas_region, "FUENTES": FUENTES})

# ---------- Herramientas de verificación ----------
_FALTA = object()


def _h(valor):
    if isinstance(valor, str):
        valor = valor.strip().lower()
    return hashlib.sha256(f"{type(valor).__name__}|{valor!r}".encode("utf-8")).hexdigest()


def _cerca(a, b, tol=1e-9):
    return math.isclose(a, b, rel_tol=1e-9, abs_tol=tol)


def _dos_decimales(x):
    return abs(x * 100 - round(x * 100)) < 1e-6


def _corto(valor, n=60):
    if type(valor).__module__ == "numpy" and getattr(valor, "shape", None) == ():
        valor = valor.item()
    texto = repr(valor)
    return texto if len(texto) <= n else texto[:n] + "…"


def _igual(a, b, tol=1e-6):
    """Compara exigiendo el mismo tipo en None/bool y tolerancia en decimales."""
    if b is None or isinstance(b, bool):
        return type(a) is type(b) and a == b
    if isinstance(b, (int, float)) and not isinstance(b, bool):
        return (isinstance(a, (int, float)) and not isinstance(a, bool)
                and math.isclose(a, b, rel_tol=1e-9, abs_tol=tol))
    if isinstance(b, (list, tuple)):
        return (type(a) is type(b) and len(a) == len(b)
                and all(_igual(x, y, tol) for x, y in zip(a, b)))
    if isinstance(b, dict):
        return (isinstance(a, dict) and set(a) == set(b)
                and all(_igual(a[k], b[k], tol) for k in b))
    return type(a) is type(b) and a == b


class _Revision:
    def __init__(self, titulo):
        self.titulo = titulo
        self.errores = 0
        print(f"── {titulo} ──")

    def ok(self, msg):
        print(f"✅ {msg}")

    def mal(self, msg):
        self.errores += 1
        print(f"❌ {msg}")

    def var(self, nombre, tipo=None):
        valor = globals().get(nombre, _FALTA)
        if valor is _FALTA:
            self.mal(f"No encuentro `{nombre}`. ¿Ejecutaste tu celda? ¿Escribiste bien el nombre?")
            return _FALTA
        if tipo is not None and not (type(valor) is tipo or (isinstance(tipo, tuple) and type(valor) in tipo)):
            esperado = tipo.__name__ if not isinstance(tipo, tuple) else " o ".join(t.__name__ for t in tipo)
            self.mal(f"`{nombre}` es de tipo {type(valor).__name__} y se esperaba {esperado}.")
            return _FALTA
        return valor

    def funcion(self, nombre):
        f = self.var(nombre)
        if f is _FALTA:
            return _FALTA
        if not callable(f):
            self.mal(f"`{nombre}` existe pero no es una función. ¿La definiste con `def`?")
            return _FALTA
        return f

    def caso(self, texto, f, args=(), kwargs=None, esperado=None, igual=None, motivo="no es lo esperado", tol=0.0051):
        """Llama a f con copias de los argumentos y compara sin mostrar el valor esperado."""
        import copy
        try:
            obtenido = f(*copy.deepcopy(args), **copy.deepcopy(kwargs or {}))
        except Exception as e:
            self.mal(f"`{texto}` lanzó {type(e).__name__}: {e}")
            return False
        bien = igual(obtenido, esperado) if igual else _igual(obtenido, esperado, tol)
        if bien:
            self.ok(f"`{texto}` funciona.")
        else:
            self.mal(f"`{texto}` devolvió {_corto(obtenido)}; {motivo}.")
        return bien

    def valor(self, nombre, esperado, tipo=None, pista="revisa el cálculo", igual=None):
        v = self.var(nombre, tipo)
        if v is _FALTA:
            return
        bien = igual(v, esperado) if igual else _igual(v, esperado)
        if bien:
            self.ok(f"`{nombre}` es correcto.")
        else:
            self.mal(f"`{nombre}` vale {_corto(v)}; {pista}.")

    def texto_limpio(self, nombre, valor):
        if valor != valor.strip():
            self.mal(f"`{nombre}` tiene espacios o saltos de línea al inicio o al final: {valor!r}")
            return False
        return True

    def predicciones(self, esperados):
        for nombre, hash_ok in esperados.items():
            v = self.var(nombre)
            if v is _FALTA:
                continue
            if _h(v) == hash_ok:
                self.ok(f"`{nombre}` es correcto.")
            else:
                self.mal(f"`{nombre}` no es correcto. Razónalo otra vez y luego compruébalo ejecutando la expresión en una celda nueva.")

    def fin(self):
        if self.errores == 0:
            print(f"🎉 ¡{self.titulo} superado!")
        else:
            cuantos = "el punto marcado" if self.errores == 1 else f"los {self.errores} puntos marcados"
            print(f"🔁 Corrige {cuantos} con ❌ y vuelve a verificar.")


def _primera_diferencia(r, nombre, tuyo, esperado):
    for i, (a, b) in enumerate(zip(tuyo, esperado)):
        if a != b:
            r.mal(f"`{nombre}` no tiene el formato pedido. La diferencia empieza en el carácter {i}: "
                  f"desde ahí tu texto dice {tuyo[i:i + 15]!r}.")
            return
    n = abs(len(esperado) - len(tuyo))
    cuantos = "1 carácter" if n == 1 else f"{n} caracteres"
    if len(tuyo) < len(esperado):
        r.mal(f"`{nombre}` está incompleto: le {'falta' if n == 1 else 'faltan'} {cuantos} al final.")
    else:
        r.mal(f"`{nombre}` tiene {cuantos} de más al final: {tuyo[len(esperado):]!r}.")


def _es_numero(x):
    return isinstance(x, (int, float, np.integer, np.floating)) and not isinstance(x, (bool, np.bool_))


def _esc(r, nombre, esperado, pista, tol=1e-6):
    v = r.var(nombre)
    if v is _FALTA:
        return
    if isinstance(v, np.ndarray) and v.shape == ():
        v = v.item()
    if not _es_numero(v):
        r.mal(f"`{nombre}` es de tipo {type(v).__name__} y se esperaba un número.")
    elif abs(float(v) - esperado) <= tol:
        r.ok(f"`{nombre}` es correcto.")
    else:
        r.mal(f"`{nombre}` vale {_corto(v.item() if hasattr(v, 'item') else v)}; {pista}.")


def _arr(r, nombre, esperado, pista, tol=1e-6, tipos=None):
    v = r.var(nombre)
    if v is _FALTA:
        return
    if not isinstance(v, np.ndarray):
        r.mal(f"`{nombre}` es de tipo {type(v).__name__} y se esperaba un array de NumPy (`np.ndarray`).")
        return
    esperado = np.array(esperado)
    if v.shape != esperado.shape:
        r.mal(f"`{nombre}` tiene forma {v.shape} y se esperaba {esperado.shape}.")
        return
    if tipos and v.dtype.kind not in tipos:
        nombres = {"b": "bool", "i": "entero", "u": "entero", "f": "decimal (float)", "U": "texto"}
        r.mal(f"`{nombre}` tiene dtype {v.dtype} y se esperaba un tipo {' o '.join(sorted({nombres[t] for t in tipos}))}.")
        return
    if v.dtype.kind in "USO" or esperado.dtype.kind in "USO":
        bien = v.tolist() == esperado.tolist()
    elif v.dtype.kind == "b" or esperado.dtype.kind == "b":
        bien = np.array_equal(v, esperado)
    else:
        bien = np.allclose(v.astype(float), esperado.astype(float), rtol=0, atol=tol, equal_nan=True)
    if bien:
        r.ok(f"`{nombre}` es correcto.")
    else:
        r.mal(f"`{nombre}` tiene la forma correcta pero sus valores no coinciden; {pista}.")




def _sin_cambios(r, *nombres):
    for n in nombres:
        actual = globals().get(n)
        original = _D[n]
        if isinstance(original, np.ndarray):
            igual = isinstance(actual, np.ndarray) and actual.shape == original.shape and np.array_equal(actual, original, equal_nan=original.dtype.kind == "f")
        else:
            igual = actual == original
        if not igual:
            r.mal(f"`{n}` cambió. No modifiques los datos originales; vuelve a ejecutar el setup.")


def _norm(x):
    if isinstance(x, np.generic):
        x = x.item()
    try:
        if pd.isna(x):
            return None
    except (TypeError, ValueError):
        pass
    if isinstance(x, pd.Timestamp):
        return str(x)
    return x


def _mismo(a, b, tol=1e-6):
    a, b = _norm(a), _norm(b)
    if a is None or b is None:
        return a is None and b is None
    if isinstance(b, (int, float)) and not isinstance(b, bool):
        return isinstance(a, (int, float)) and not isinstance(a, bool) and abs(a - b) <= tol
    return str(a) == str(b)


def _ser(r, nombre, valores, pista, indice=None, tol=1e-6):
    v = r.var(nombre)
    if v is _FALTA:
        return
    if not isinstance(v, pd.Series):
        r.mal(f"`{nombre}` es de tipo {type(v).__name__} y se esperaba una Series de pandas.")
        return
    if len(v) != len(valores):
        r.mal(f"`{nombre}` tiene {len(v)} elementos y se esperaban {len(valores)}.")
        return
    if indice is not None and [_norm(i) for i in v.index] != indice:
        r.mal(f"Las etiquetas (índice) de `{nombre}` no son las esperadas; {pista}.")
        return
    if all(_mismo(a, b, tol) for a, b in zip(v.tolist(), valores)):
        r.ok(f"`{nombre}` es correcto.")
    else:
        r.mal(f"`{nombre}` tiene el largo correcto pero sus valores no coinciden; {pista}.")


def _df(r, nombre, columnas, filas, pista, indice=None, tol=1e-6):
    v = r.var(nombre)
    if v is _FALTA:
        return
    if not isinstance(v, pd.DataFrame):
        r.mal(f"`{nombre}` es de tipo {type(v).__name__} y se esperaba un DataFrame de pandas.")
        return
    cols = [str(c) for c in v.columns]
    if cols != columnas:
        faltan = [c for c in columnas if c not in cols]
        sobran = [c for c in cols if c not in columnas]
        if faltan or sobran:
            r.mal(f"A `{nombre}` le faltan las columnas {faltan} y le sobran {sobran}." if faltan and sobran else
                  (f"A `{nombre}` le faltan las columnas {faltan}." if faltan else f"En `{nombre}` sobran las columnas {sobran}."))
        else:
            r.mal(f"`{nombre}` tiene las columnas correctas pero en otro orden.")
        return
    if len(v) != len(filas):
        r.mal(f"`{nombre}` tiene {len(v)} filas y se esperaban {len(filas)}.")
        return
    if indice is not None and [_norm(i) for i in v.index] != indice:
        r.mal(f"Las etiquetas de fila (índice) de `{nombre}` no son las esperadas; {pista}.")
        return
    bien = all(_mismo(a, b, tol) for fila_v, fila_e in zip(v.itertuples(index=False), filas) for a, b in zip(fila_v, fila_e))
    if bien:
        r.ok(f"`{nombre}` es correcto.")
    else:
        r.mal(f"`{nombre}` tiene la forma correcta pero sus valores no coinciden; {pista}.")


def _sin_cambios_df(r, *nombres):
    for n in nombres:
        actual = globals().get(n)
        if not (isinstance(actual, (pd.DataFrame, pd.Series)) and actual.equals(_D[n])):
            r.mal(f"`{n}` cambió. No modifiques los datos originales; vuelve a cargarlos o ejecuta de nuevo el setup.")

def _hex(c):
    return mcolors.to_hex(c).lower()


def _texto(a, b):
    return isinstance(a, str) and a.strip().lower() == b.strip().lower()


def _grafico(r, nombre):
    ax = r.var(nombre)
    if ax is _FALTA:
        return None
    if not isinstance(ax, mpl.axes.Axes):
        r.mal(f"`{nombre}` es de tipo {type(ax).__name__} y se esperaba un eje de Matplotlib (lo que devuelve `plt.subplots()`).")
        return None
    return ax


def _rotulos(r, nombre, ax, titulo=None, xlabel=None, ylabel=None):
    titulo_actual = next((t for t in (ax.get_title(loc=l) for l in ("left", "center", "right")) if t.strip()), "")
    for que, genero, obtenido, esperado in (("el título", "correcto", titulo_actual, titulo),
                                            ("la etiqueta del eje x", "correcta", ax.get_xlabel(), xlabel),
                                            ("la etiqueta del eje y", "correcta", ax.get_ylabel(), ylabel)):
        if esperado is None:
            continue
        if _texto(obtenido, esperado):
            r.ok(f"En `{nombre}`, {que} es {genero}.")
        elif not obtenido.strip():
            r.mal(f"A `{nombre}` le falta {que}.")
        else:
            r.mal(f"En `{nombre}`, {que} dice {obtenido!r}; revisa el texto pedido.")


def _barras(ax):
    """Rectángulos de barras (sin el fondo del eje), en orden de dibujo."""
    return [p for p in ax.patches if isinstance(p, mpl.patches.Rectangle)]


def _cerca_lista(a, b, tol=1e-6):
    return len(a) == len(b) and all(abs(float(x) - float(y)) <= tol for x, y in zip(a, b))


def _formato(ax, eje, valor):
    fmt = (ax.yaxis if eje == "y" else ax.xaxis).get_major_formatter()
    return fmt(valor, 0)


def _hx(texto):
    """Huella del texto exacto (distingue mayúsculas, espacios y signos)."""
    return hashlib.sha256(str(texto).encode("utf-8")).hexdigest()


_CASOS_SOLES = [13_120_400.0, 4_870_000.0, 1_000_000.0, 356_700.0, 1_000.0, 999.0, 0.0, -45_600.0, -2_340_000.0]
_CASOS_PCT = [0.125, 0.3348, -0.03, 0.0, 1.0]
_CASOS_CITA = [dict(FUENTES[1]), dict(FUENTES[2]),
               {"autores": "  Pérez, A. ", "anio": 2023, "titulo": " Informe anual ", "tipo": "Informe", "editor": "Banco X ", "url": " "}]
_H_SOLES = ["018b10cc9bd667f17b7f24d4a2bf95479b2dc3a7fc8d336ef416015ce88dd0dd", "2bc28ccbcffdb6bb1f85204d9ae3071137cbeed38e402d21578bc146c45f6dba", "7354d5f7d32a7443ee0386554cb4eada8f25c0b4a7279ebb0a01632caa734196", "29f328369710b59a4e7d33d17ec50fa684502411cb1d32d2af075accfc25502b", "3a552379ed3d664c9c228577c431d40b24a67926fead4ebbd2adde9280185f1a", "a930a29bb6c5a57add0ec86b27c25b57bf84458724cd8b6b494274cc07f19a68", "2a960d469ff1f56c6c19d9914fc0b114b041351b1fc36bda282823be48ce4a69", "dc726242b74a1113708feb7073c229a324628d78aefbd3398310541d413a4723", "926d4c4d6be942b7e1515c714f37bbf96bd42d993b68f17e4af95ef84e711005"]
_H_PCT = ["1628e52315081f4f8983600f6729532aa0b6f387c9fc8404985e5e296ebb43ae", "5a4538ca6a93197f19dfce70b25152e132543dbe8d495ccaf46e6f6c36b66fce", "0711d8e08f19933e7f4b9a8149daf1e1816a3d59a83191546480d9773271029c", "d12aabacd48e53a21a42c333a6de5610eb5eaf397ce6f54617cd72cfbf57eab4", "5bbd79158e6af6cc0f3d8cc0ed30dff8b12096cf8b03298ce5c0e6b48217520a"]
_H_CITA = ["451b20592e8b17b9bdac30700fd33c794c06a646c85cbc651bae1b0dde588f48", "6b3ef48cf8d1d685f61c3f5b986fb179008b88a822ee8442a548c2f3d801f04a", "18991666ebc4500fc11568e8d936381165d9a02896c8952367e5385d276e1b53"]
_H_TABLA = [
        [
            "aaf2d054d5f750cbf297a4ecd8c58f861a34a48ff7159ce6cd08f986f6a0c0eb",
            "b782dc97207c9b1d7fd4b25b98c75564de6cb1caf37f0e0bef17b4028ed70df0",
            "2b559006371336755c02ca25eaf202e664330a94562d580a5711f8df2dce7d9f",
            "4932d2a55bea6483cf95200ed100d5fc4f7315ba76c23f6d4435a32bdc67d494"
        ],
        [
            "6f9f01a63d42c99d94a4a14ac7fa7e7b1bedb667525c0defa86b56329189b1f3",
            "f62841c36f7065f1003edb69d874efcd8e2d5f1e2298e8eb208bade4e4618ff6",
            "bc730c1e836c7a76386d8801a5caebbda33eb4bd59cbd3e85822809f2ef9a72d",
            "70a0bdb90be29ee513b4ea8cce1e578bdead48e41da98f85d0fa0091cc6a3b0f"
        ],
        [
            "eb5d6ff7db3e43341f371e480b9db34c5178dad17d4141cf9aca050a188a8f2c",
            "d83c7760fe6f66fde26a2a893f8cb22a2107281c998341479fffb1866e3cdac8",
            "beb926f96e18dd7277c246a3213c747d252ad075eded385bafc3f54fb5c345aa",
            "276998099f5a967eaa9887a990afe9033dbb9c9dc300366689cfad98ddca97ee"
        ],
        [
            "fed34f0449c27518a2b7c2892633fd9ba80365ff0e5afbd6a80c78afa8877358",
            "e4beae1d5667e4a3ca674b286fa1f017fc8ad9f40b9ed9894271dd53613be4a6",
            "6e6a9ed75eb54ec61f1f4e3530de9bc2a84a1e9a2ac04f285397792f15de4f87",
            "47fe869b2288cba348927e3e23ac32a9eded595eae7d0b3a77cbd134cd201b3e"
        ],
        [
            "43161d62ec62badc5f56fb882ec2873b2220bd8a4524ada4b5e7f549995e99f7",
            "c2f3c783a51754b87710ad87c1f0da32757a5b3d490fd64c55fb813a0e8f8012",
            "2eef30aa55056339f4745741bf5b2267ed0fe6615c2df270d13a284a2a5edc65",
            "fe70f48814d66c482787d599fece98de4642c38ddab3e7325ede33877bc9c050"
        ],
        [
            "c9b3c38247f744e17dd26fda097d6a9ba9332586b6bdaa038bf8f313a863f2b8",
            "03980f6fda9f9470730b3236d116c7e088fc46de1079b84f086841aeadf199fc",
            "4fcfb970c1b93c68e91f50df899b9f64d7b808d6066c653e131155f16d7b28b9",
            "5bbd79158e6af6cc0f3d8cc0ed30dff8b12096cf8b03298ce5c0e6b48217520a"
        ]
    ]
_H_REFS = ["6b3ef48cf8d1d685f61c3f5b986fb179008b88a822ee8442a548c2f3d801f04a", "44df2806ffd4e063d40dc1b87e798504515ca67fae1f4a67b9ba99fe3ce23ae9", "451b20592e8b17b9bdac30700fd33c794c06a646c85cbc651bae1b0dde588f48"]
_H_KPI = ["03980f6fda9f9470730b3236d116c7e088fc46de1079b84f086841aeadf199fc", "4fcfb970c1b93c68e91f50df899b9f64d7b808d6066c653e131155f16d7b28b9", "43161d62ec62badc5f56fb882ec2873b2220bd8a4524ada4b5e7f549995e99f7"]


def _probar(r, nombre, casos, huellas, motivo):
    f = r.funcion(nombre)
    if f is _FALTA:
        return
    for (texto, args, kwargs), huella in zip(casos, huellas):
        try:
            res = f(*copy.deepcopy(args), **copy.deepcopy(kwargs))
        except Exception as ex:
            r.mal(f"`{texto}` lanzó {type(ex).__name__}: {ex}")
            continue
        if not isinstance(res, str):
            r.mal(f"`{texto}` devolvió {type(res).__name__} y se esperaba un texto (str).")
        elif _hx(res) != huella:
            r.mal(f"`{texto}` devolvió {res!r}; {motivo}.")
        else:
            r.ok(f"`{texto}` funciona.")


def _texto_con_numero(t):
    return isinstance(t, str) and any(ch.isdigit() for ch in t)


def check_ejercicio_1():
    r = _Revision("Ejercicio 1 · Parte A")
    _sin_cambios_df(r, "ventas_region")
    motivo = "revisa el formato pedido (decimales, coma, espacios y signo)"
    _probar(r, "formato_soles", [(f"formato_soles({x!r})", (x,), {}) for x in _CASOS_SOLES], _H_SOLES, motivo)
    _probar(r, "formato_pct", [(f"formato_pct({x!r})", (x,), {}) for x in _CASOS_PCT], _H_PCT, motivo)
    r.fin()
    r = _Revision("Ejercicio 1 · Parte B")
    r.predicciones({
        "pred_formato_5": "a07a610e437d528b1ad59e0eb53add738f31f1137e19eb04f4417d37725b568c",
    })
    r.fin()


def check_ejercicio_2():
    r = _Revision("Ejercicio 2 · Parte A")
    t = r.var("tabla_ejecutiva")
    cols = ["Región", "Ventas 2024", "Crecimiento", "Participación"]
    if t is not _FALTA:
        if not isinstance(t, pd.DataFrame) or [str(c) for c in t.columns] != cols:
            r.mal(f"`tabla_ejecutiva` debería ser un DataFrame con las columnas {cols}, en ese orden.")
        elif len(t) != len(_H_TABLA):
            r.mal(f"`tabla_ejecutiva` tiene {len(t)} filas y se esperaban {len(_H_TABLA)}: una por región más la fila Total.")
        elif [_hx(x) for x in t["Región"]] != [f[0] for f in _H_TABLA]:
            r.mal("Las regiones de `tabla_ejecutiva` deberían ir de mayor a menor venta en 2024, con la fila \"Total\" al final.")
        elif not all(isinstance(x, str) for x in t[cols[1:]].to_numpy().ravel()):
            r.mal("Las columnas con números de `tabla_ejecutiva` deberían ser texto ya formateado con tus funciones.")
        else:
            malas = [(i, cols[j]) for i, fila in enumerate(t.astype(str).values.tolist()) for j, x in enumerate(fila) if _hx(x) != _H_TABLA[i][j]]
            if malas:
                i, c = malas[0]
                r.mal(f"En `tabla_ejecutiva`, el valor de la fila {i} en la columna {c!r} no coincide: usa `formato_soles` y `formato_pct`; la participación es sobre el total de 2024.")
            elif list(t.index) != list(range(len(_H_TABLA))):
                r.mal("Reinicia el índice de `tabla_ejecutiva` (`reset_index(drop=True)`) para que no muestre números sueltos.")
            else:
                r.ok("`tabla_ejecutiva` está lista para una diapositiva.")
    r.fin()
    r = _Revision("Ejercicio 2 · Parte B")
    r.predicciones({
        "pred_suma_participacion": "848b31cf3d6ab128856b68e3aa32677dfc6cb6e01c2d2ee051bc4144ace656a3",
    })
    r.fin()


def check_ejercicio_3():
    r = _Revision("Ejercicio 3 · Parte A")
    m = r.var("mensaje_principal", str)
    if m is not _FALTA:
        palabras = m.split()
        if len(palabras) > 30:
            r.mal(f"`mensaje_principal` tiene {len(palabras)} palabras: resúmelo en 30 o menos.")
        elif len(palabras) < 8:
            r.mal("`mensaje_principal` es demasiado corto: debe decir qué pasa y qué hacer.")
        elif not _texto_con_numero(m):
            r.mal("`mensaje_principal` debería incluir al menos un número que lo respalde.")
        elif m.strip().lower().startswith(("en este", "este análisis", "el presente", "a continuación")):
            r.mal("Empieza `mensaje_principal` con la conclusión, no con una introducción.")
        else:
            r.ok("`mensaje_principal` es breve, concreto y tiene un número.")
    p = r.var("propuesta", dict)
    claves = ["contexto", "hallazgos", "recomendacion", "impacto", "riesgos", "siguiente_paso"]
    if p is not _FALTA:
        if list(p) != claves:
            r.mal(f"`propuesta` debería tener las claves {claves}, en ese orden.")
        elif not isinstance(p["hallazgos"], list) or len(p["hallazgos"]) != 3 or not all(_texto_con_numero(h) for h in p["hallazgos"]):
            r.mal("`propuesta[\"hallazgos\"]` debería ser una lista de 3 hallazgos, cada uno con un número.")
        elif not all(isinstance(p[k], str) and len(p[k].split()) >= 5 for k in claves if k != "hallazgos"):
            r.mal("Cada parte de `propuesta` (salvo los hallazgos) debería ser una frase completa de al menos 5 palabras.")
        elif not _texto_con_numero(p["impacto"]):
            r.mal("`propuesta[\"impacto\"]` debería estimar el impacto con un número (y su supuesto).")
        else:
            r.ok("`propuesta` tiene la estructura completa.")
    r.fin()
    r = _Revision("Ejercicio 3 · Parte B")
    r.predicciones({
        "pred_recomendacion_va": "7b31f6c4a73248b396007e49b2a34d95527fbd0be0435d32457ab1284b21ff9e",
    })
    r.fin()


def check_ejercicio_4():
    r = _Revision("Ejercicio 4 · Parte A")
    ax = _grafico(r, "ax_slide")
    if ax is not None:
        v = ventas_region.assign(c=ventas_region["ventas_2024"] / ventas_region["ventas_2023"] - 1).sort_values("c")
        barras = _barras(ax)
        titulo = next((t for t in (ax.get_title(loc=l) for l in ("left", "center", "right")) if t.strip()), "")
        textos_fig = [t.get_text() for t in ax.figure.texts]
        if len(barras) != len(v) or not _cerca_lista([b.get_width() for b in barras], v["c"].tolist(), 1e-9):
            r.mal("`ax_slide` debería tener barras horizontales con el crecimiento de cada región (como proporción), de menor a mayor.")
        elif [_hex(b.get_facecolor()) for b in barras] != [GRIS] * (len(v) - 1) + [AZUL]:
            r.mal("En `ax_slide`, destaca en `AZUL` la región que más creció y deja el resto en `GRIS`.")
        elif not _texto_con_numero(titulo) or len(titulo.split()) < 4:
            r.mal("El título de `ax_slide` debería ser la conclusión, con un número.")
        elif not any("fuente" in t.lower() for t in textos_fig):
            r.mal("Agrega una nota al pie con la fuente usando `fig_slide.text(...)` (que diga \"Fuente: ...\").")
        else:
            r.ok("`ax_slide` está listo para una diapositiva.")
    if not os.path.exists("slide_crecimiento.png"):
        r.mal("No encuentro `slide_crecimiento.png`. Guárdalo con `fig_slide.savefig(\"slide_crecimiento.png\", dpi=200, bbox_inches=\"tight\")`.")
    else:
        alto, ancho = plt.imread("slide_crecimiento.png").shape[:2]
        if ancho < 1200:
            r.mal(f"`slide_crecimiento.png` mide {ancho} píxeles de ancho: guárdalo con `dpi=200` para que se lea bien proyectado.")
        else:
            r.ok("`slide_crecimiento.png` se guardó con buena resolución.")
    r.fin()
    r = _Revision("Ejercicio 4 · Parte B")
    r.predicciones({
        "pred_ancho_px": "f95105a0c4551a18ac203bca11f6737b9e5abb9b21377266fe221d216e099a98",
    })
    r.fin()


def check_ejercicio_5():
    r = _Revision("Ejercicio 5 · Parte A")
    _sin_cambios(r, "FUENTES")
    f = r.funcion("cita_apa")
    if f is not _FALTA:
        _probar(r, "cita_apa", [(f"cita_apa(**{{... \"titulo\": {c['titulo']!r} ...}})", (), c) for c in _CASOS_CITA], _H_CITA,
                "revisa el orden, los paréntesis, los corchetes, los asteriscos del título, los puntos y los espacios sobrantes; sin URL, termina en el editor")
    refs = r.var("referencias", list)
    if refs is not _FALTA:
        if len(refs) != len(_H_REFS):
            r.mal(f"`referencias` tiene {len(refs)} elementos y se esperaban {len(_H_REFS)}, una por fuente.")
        elif [_hx(x) for x in refs] != _H_REFS:
            r.mal("`referencias` debería tener la cita de cada fuente de `FUENTES`, en orden alfabético (sin distinguir mayúsculas).")
        else:
            r.ok("`referencias` es correcta.")
    r.fin()
    r = _Revision("Ejercicio 5 · Parte B")
    r.predicciones({
        "pred_basta_citar": "e675bdd897ba87a607b7c344f97a5152cda452c1b1641515ea9436fd35397ada",
    })
    r.fin()


def check_reto():
    r = _Revision("Reto final")
    fig = r.var("fig_resumen")
    if fig is not _FALTA:
        if not isinstance(fig, mpl.figure.Figure):
            r.mal("`fig_resumen` debería ser una figura de Matplotlib.")
        else:
            textos = {_hx(t.get_text().strip()) for ax in fig.axes for t in ax.texts} | {_hx(t.get_text().strip()) for t in fig.texts}
            con_barras = [ax for ax in fig.axes if _barras(ax)]
            if len(fig.axes) < 4:
                r.mal(f"`fig_resumen` tiene {len(fig.axes)} ejes y se esperaban al menos 4: tres indicadores y un gráfico.")
            elif [h for h in _H_KPI if h not in textos]:
                r.mal("Los indicadores de `fig_resumen` deberían mostrar, cada uno en su propio `ax.text`, las ventas totales de 2024 y su crecimiento (con tus funciones de formato) y el nombre de la región que más creció.")
            elif not con_barras:
                r.mal("`fig_resumen` debería incluir un gráfico de barras.")
            else:
                r.ok("`fig_resumen` resume el año en una página.")
    if not os.path.exists("resumen_ejecutivo.png"):
        r.mal("No encuentro `resumen_ejecutivo.png`: guárdalo con `dpi=200`.")
    else:
        r.ok("`resumen_ejecutivo.png` está guardado.")
    r.fin()


def check_pro():
    r = _Revision("Nivel pro")
    largo = []
    for f in ventas_region.itertuples(index=False):
        largo += [[f.region, 2023, f.ventas_2023], [f.region, 2024, f.ventas_2024]]
    largo.sort(key=lambda x: (x[0], x[1]))
    _df(r, "ventas_largo", ["region", "anio", "ventas"], largo, "una fila por región y año, ordenada por región y año, con `anio` como entero", tol=1e-6)
    if not os.path.exists("ventas_region_largo.csv"):
        r.mal("No encuentro `ventas_region_largo.csv`.")
    else:
        leido = pd.read_csv("ventas_region_largo.csv")
        if list(leido.columns) != ["region", "anio", "ventas"] or len(leido) != len(largo):
            r.mal("`ventas_region_largo.csv` debería tener las columnas region, anio y ventas, sin índice (`index=False`).")
        else:
            r.ok("`ventas_region_largo.csv` está listo para Power BI.")
    r.fin()


print("✅ Setup listo. Datos generados, estilo aplicado y verificadores cargados.")

### 📦 Tus datos de hoy
`ventas_region`: las ventas (en soles) de una cadena de tiendas en cinco regiones, en 2023 y 2024. Es el resultado final de un análisis: hoy no calculas nada nuevo, **lo comunicas**.

`FUENTES`: una lista con los datos de tres fuentes para citar.

In [ ]:
print(ventas_region, "\n")
for fuente in FUENTES:
    print(fuente)

---
## 1. Números que se leen en un segundo

### 📘 Concepto
Quien decide no lee `13120400.0`: lee **S/ 13,1 M**. Reglas para presentar cifras:
- redondea a lo que importa para decidir (una cifra decimal en millones, miles enteros);
- usa la unidad (S/, %) y el formato local: en Perú, coma decimal;
- un signo menos claro para las caídas;
- escribe el formato en **funciones**, para que todo el informe use el mismo criterio.

Los f-strings formatean con `:.1f` (un decimal); después se cambia el punto por coma con `.replace(".", ",")`.

In [ ]:
valor_ej = 2_456_789.0
print(f"{valor_ej / 1_000_000:.1f}".replace(".", ",") + " M")
print(f"{0.0734 * 100:.1f}".replace(".", ",") + " %")
print(round(356.7), abs(-12))

### ✍️ Tu turno · Ejercicio 1: tus funciones de formato
**Parte A.**
1. `formato_soles(x)`: devuelve un texto según el valor absoluto de `x`:
   - desde 1 000 000: `"S/ 13,1 M"` (millones con un decimal y coma);
   - desde 1 000: `"S/ 357 mil"` (miles redondeados, sin decimales);
   - menos de 1 000: `"S/ 999"` (redondeado, sin decimales).
   Si `x` es negativo, agrega un `-` al inicio: `"-S/ 46 mil"`.
2. `formato_pct(x)`: recibe una proporción y devuelve el porcentaje con un decimal, coma y un espacio antes del signo: `0.125` → `"12,5 %"`, `-0.03` → `"-3,0 %"`.

**Parte B.** Predice **sin ejecutar**: ¿qué devuelve `formato_pct(0.05)`? Guárdalo en `pred_formato_5` (un texto).

In [ ]:
# Tu código aquí


In [ ]:
# ✅ Verificar
check_ejercicio_1()

<details><summary>💡 Pista 1</summary>

Trabaja con `a = abs(x)` y decide el signo aparte: `signo = "-" if x < 0 else ""`.
</details>

<details><summary>💡 Pista 2</summary>

Para los miles: `f"{round(a / 1000)} mil"`. Arma el resultado con `f"{signo}S/ {cuerpo}"`.
</details>

---
## 2. Una tabla para una diapositiva

### 📘 Concepto
Una tabla para decisores no es un `print(df)`:
- pocas columnas, con nombres en lenguaje de negocio (`"Ventas 2024"`, no `ventas_2024`);
- ordenada por lo que importa (normalmente, de mayor a menor);
- números ya formateados y una fila de **Total** para dar contexto;
- sin el índice numérico de pandas.

`df.rename(columns={...})` cambia nombres, `.map(funcion)` aplica tu formato a una columna y `pd.concat` agrega la fila del total.

In [ ]:
tabla_ej = pd.DataFrame({"tienda": ["A", "B"], "ventas": [1_500_000.0, 820_000.0]})
total_ej = pd.DataFrame({"tienda": ["Total"], "ventas": [tabla_ej["ventas"].sum()]})
tabla_ej = pd.concat([tabla_ej, total_ej], ignore_index=True)
tabla_ej["ventas"] = tabla_ej["ventas"].map(lambda v: f"{v:,.0f}")
print(tabla_ej.rename(columns={"tienda": "Tienda", "ventas": "Ventas (S/)"}).to_string(index=False))

### ✍️ Tu turno · Ejercicio 2: la tabla ejecutiva
**Parte A.** `tabla_ejecutiva`: un DataFrame con las columnas `Región`, `Ventas 2024`, `Crecimiento` (2024 frente a 2023) y `Participación` (de las ventas de 2024 sobre el total de 2024), con:
- las regiones de mayor a menor venta en 2024 y una última fila `Total` (su participación es el 100 %);
- los valores formateados con `formato_soles` y `formato_pct`;
- el índice reiniciado (0, 1, 2...).

Muéstrala con `print(tabla_ejecutiva.to_string(index=False))`.

**Parte B.** Predice **sin ejecutar**: ¿cuánto suma, en porcentaje, la participación de las cinco regiones? Guárdalo en `pred_suma_participacion` (un número decimal).

In [ ]:
# Tu código aquí


In [ ]:
# ✅ Verificar
check_ejercicio_2()

<details><summary>💡 Pista 1</summary>

Calcula primero las columnas numéricas (crecimiento y participación) en una copia de `ventas_region`, ordena, agrega la fila Total y recién al final formatea.
</details>

<details><summary>💡 Pista 2</summary>

Para la fila Total: crecimiento = total 2024 / total 2023 − 1, y participación = 1.0. Termina con `.reset_index(drop=True)`.
</details>

---
## 3. La propuesta: la conclusión primero

### 📘 Concepto
Un informe académico va de los datos a la conclusión. Una **propuesta de negocio** va al revés (principio de la pirámide): primero la **respuesta**, después lo que la respalda. Quien decide tiene poco tiempo; si solo lee la primera línea, debe llevarse lo importante.

Una estructura que funciona:
1. **Contexto**: qué problema o pregunta hay (una o dos frases).
2. **Hallazgos**: tres, cada uno con un número.
3. **Recomendación**: qué hacer, concreto.
4. **Impacto**: cuánto vale, con sus supuestos.
5. **Riesgos**: qué podría salir mal o qué no sabemos.
6. **Siguiente paso**: qué se necesita decidir o probar ahora.

El **mensaje principal** resume todo en una o dos frases: qué pasa, qué hacer y un número.

In [ ]:
malo_ej = "En este análisis revisamos los datos de ventas de varios años y aplicamos distintas técnicas."
bueno_ej = "La tienda B creció 25 %: ampliar su horario podría sumar S/ 0,3 M al año."
for texto in (malo_ej, bueno_ej):
    print(len(texto.split()), "palabras |", texto)

### ✍️ Tu turno · Ejercicio 3: tu propuesta en una página
Usa los resultados de `ventas_region` y tu `tabla_ejecutiva`.

**Parte A.**
1. `mensaje_principal`: entre 8 y 30 palabras, con al menos un número, que empiece por la conclusión.
2. `propuesta`: un diccionario con las claves `contexto`, `hallazgos`, `recomendacion`, `impacto`, `riesgos` y `siguiente_paso`, en ese orden. `hallazgos` es una lista de 3 textos con un número cada uno; el resto son frases completas, y `impacto` incluye un número.

**Parte B.** Responde en `pred_recomendacion_va` con `"principio"` o `"final"`: en una propuesta para decisores, ¿dónde va la respuesta principal?

In [ ]:
# Tu código aquí


In [ ]:
# ✅ Verificar
check_ejercicio_3()

<details><summary>💡 Pista 1</summary>

Mira qué región crece más y cuál pesa más. Un buen mensaje combina las dos cosas: dónde está el negocio hoy y dónde está creciendo.
</details>

<details><summary>💡 Pista 2</summary>

Si la recomendación implica un costo que no tienes, dilo como supuesto en `impacto` ("si una tienda nueva vendiera como el promedio de Oriente...").
</details>

---
## 4. El gráfico de la diapositiva

### 📘 Concepto
Un gráfico para una diapositiva cumple todo lo de las sesiones 15 y 16 y además:
- el **título es la conclusión** (con un número), no la descripción de los ejes;
- un solo mensaje: destaca lo importante en un color y deja el resto en gris;
- la **fuente** en una nota al pie: `fig.text(0.01, 0.01, "Fuente: ...", fontsize=9)`;
- se exporta con buena resolución: `fig.savefig("archivo.png", dpi=200, bbox_inches="tight")`. El ancho en píxeles es el ancho en pulgadas (`figsize`) por `dpi`.

In [ ]:
fig_ej, ax_ej = plt.subplots(figsize=(6, 2.5))
ax_ej.barh(["B", "A"], [0.05, 0.12], color=[GRIS, AZUL])
ax_ej.xaxis.set_major_formatter(mpl.ticker.PercentFormatter(1.0, decimals=0))
ax_ej.set_title("A crece más del doble que B")
fig_ej.text(0.01, -0.05, "Fuente: datos de ejemplo.", fontsize=9, color=TINTA_2)
plt.show()

### ✍️ Tu turno · Ejercicio 4: el crecimiento por región
**Parte A.**
1. `fig_slide, ax_slide = plt.subplots(figsize=(8, 4.5))`: barras horizontales con el crecimiento de cada región (como proporción, eje en porcentaje), de menor a mayor (la mayor arriba), la región que más creció en `AZUL` y el resto en `GRIS`.
2. Un título que sea la conclusión, con un número, y una nota al pie con `fig_slide.text(...)` que diga `Fuente: ...`.
3. Guárdalo como `slide_crecimiento.png` con `dpi=200` y `bbox_inches="tight"`.

**Parte B.** Predice **sin ejecutar**: con `figsize=(8, 4.5)` y `dpi=200`, ¿cuántos píxeles de ancho tiene la imagen (sin `bbox_inches`)? Guárdalo en `pred_ancho_px` (un entero).

In [ ]:
# Tu código aquí


In [ ]:
# ✅ Verificar
check_ejercicio_4()

<details><summary>💡 Pista 1</summary>

Calcula el crecimiento en una Series con la región como índice y ordénala con `sort_values()`.
</details>

<details><summary>💡 Pista 2</summary>

Colores: `[AZUL if region == crecimiento.idxmax() else GRIS for region in crecimiento.index]`. Para el eje, `mpl.ticker.PercentFormatter(1.0)`.
</details>

---
## 5. Referencias y un notebook limpio

### 📘 Concepto
Todo dato que no generaste tú se **cita**. Un formato habitual (APA) para datasets es:

`Autores (Año). *Título* [Tipo]. Editor. URL`

(los asteriscos ponen el título en cursiva en Markdown). Citar no alcanza: revisa también la **licencia** de cada fuente, que dice si puedes usar, modificar o publicar los datos.

Un **notebook limpio** es el que otra persona puede ejecutar de principio a fin (*Reiniciar y ejecutar todo*) y entender:
- imports y constantes (rutas, fechas de corte, semillas) arriba;
- funciones para lo que se repite, con nombres claros;
- sin celdas de prueba sueltas ni resultados que dependan del orden en que ejecutaste;
- títulos Markdown que cuentan la historia y conclusiones escritas junto a cada resultado;
- una lista de referencias al final.

In [ ]:
def saludo_ej(nombre, cargo=None):
    texto = f"Hola, {nombre.strip()}."
    return texto + (f" ({cargo.strip()})" if cargo and cargo.strip() else "")

print(saludo_ej(" Ana "), "|", saludo_ej("Luis", "analista"), "|", saludo_ej("Eva", "  "))

### ✍️ Tu turno · Ejercicio 5: tus referencias
**Parte A.**
1. `cita_apa(autores, anio, titulo, tipo, editor, url=None)`: devuelve la cita con el formato del concepto: `Autores (Año). *Título* [Tipo]. Editor. URL`. Quita los espacios sobrantes de los textos; si `url` es `None` o está vacía, la cita termina en `Editor.`
2. `referencias`: la lista de citas de las fuentes de `FUENTES` (usa `cita_apa(**fuente)`), en orden alfabético sin distinguir mayúsculas (`sorted(..., key=str.lower)`).

**Parte B.** Responde en `pred_basta_citar` con `"sí"` o `"no"`: si citas bien un dataset, ¿ya puedes publicarlo en tu repositorio?

In [ ]:
# Tu código aquí


In [ ]:
# ✅ Verificar
check_ejercicio_5()

<details><summary>💡 Pista 1</summary>

Copia la idea de `saludo_ej`: arma la parte fija con un f-string y agrega la URL solo si existe y no está vacía.
</details>

<details><summary>💡 Pista 2</summary>

`texto = f"{autores.strip()} ({anio}). *{titulo.strip()}* [{tipo}]. {editor.strip()}."`
</details>

---
## 🏋️ Reto final: el resumen ejecutivo en una página
Crea `fig_resumen` con una fila de **tres indicadores** arriba y un **gráfico de barras** abajo:

```python
fig_resumen = plt.figure(figsize=(10, 6))
rejilla = fig_resumen.add_gridspec(2, 3, height_ratios=[1, 2.5])
ax_kpi = [fig_resumen.add_subplot(rejilla[0, i]) for i in range(3)]
ax_graf = fig_resumen.add_subplot(rejilla[1, :])
```

1. En cada `ax_kpi`, sin ejes (`ax.axis("off")`), escribe con un `ax.text(...)` el valor del indicador y con otro, su etiqueta: las ventas totales de 2024 (con `formato_soles`), su crecimiento frente a 2023 (con `formato_pct`) y la región que más creció.
2. En `ax_graf`, las ventas de 2024 por región en barras, con énfasis donde apunta tu recomendación.
3. Un título general con `fig_resumen.suptitle(mensaje_principal, ...)` y la fuente al pie.
4. Guárdalo como `resumen_ejecutivo.png` con `dpi=200`.

In [ ]:
# Tu código aquí


In [ ]:
# ✅ Verificar
check_reto()

<details><summary>💡 Pista 1</summary>

Para un indicador: `ax.text(0, 0.6, valor, fontsize=24, fontweight="bold")` y `ax.text(0, 0.2, etiqueta, fontsize=11, color=TINTA_2)`.
</details>

<details><summary>💡 Pista 2</summary>

Si el título es largo, usa `fontsize=13` y `wrap=True`, o parte el texto con `\n`.
</details>

---
## 🚀 Nivel pro (opcional): datos listos para Power BI
Power BI trabaja mejor con tablas en **formato largo**: una fila por combinación de dimensiones (región y año) y una columna por medida. Crea `ventas_largo` con las columnas `region`, `anio` (entero) y `ventas`, ordenada por región y año (usa `melt`), y guárdala como `ventas_region_largo.csv` sin índice. Así, en Power BI, el año se vuelve un filtro y no hace falta una medida por año.

In [ ]:
# Tu código aquí


In [ ]:
# ✅ Verificar
check_pro()

---
## 🧱 Avance del proyecto: P4 completo · dashboard, recomendaciones y repositorio final

**Qué hacer**
1. **Dashboard en Power BI** (`dashboard/`), conectado a las tablas de tu base SQL de S25 o a CSV en formato largo:
   - página 1, **resumen**: 3 o 4 indicadores (por ejemplo, reclamos totales, reclamos por cada 10 000 clientes, porcentaje resuelto a favor y variación anual) y la evolución en el tiempo de S26;
   - página 2, **dónde está el problema**: ranking normalizado de entidades o regiones y el reparto por motivo;
   - página 3, **segmentos y recomendación**: los segmentos de S24 y el impacto estimado de S23 con sus supuestos;
   - filtros por año, producto y región; títulos que dicen la conclusión; la fuente en cada página.
   Guarda el `.pbix` (si pesa poco) y capturas de cada página en `dashboard/`.
2. **Recomendaciones**: responde la pregunta 6 del proyecto con la estructura del ejercicio 3: mensaje principal, 3 hallazgos con números, recomendación, impacto con supuestos, riesgos y siguiente paso.
3. **Notebooks limpios**: reinicia y ejecuta todo cada notebook de `notebooks/`, borra celdas de prueba y agrega conclusiones en Markdown.
4. **README final**: problema, fuentes con cita y licencia, cómo reproducir (orden de los notebooks y dependencias), hallazgos, modelo (P3), segmentos, recomendación, capturas del dashboard, limitaciones y referencias.
5. **Presentación final** (8 a 10 diapositivas): 1) título y mensaje principal; 2) la pregunta y por qué importa; 3) datos y fuentes; 4 a 6) tres hallazgos, un gráfico por diapositiva; 7) modelo o segmentos, en lenguaje de negocio; 8) recomendación e impacto con supuestos; 9) riesgos y siguiente paso; 10) contacto y enlace al repositorio. Ensáyala en 10 minutos.

**Por qué lo haría un analista**
El trabajo técnico solo se ve si se puede recorrer: un dashboard para explorar, una recomendación para decidir y un repositorio para verificar. Es lo que mira quien revisa tu portafolio.

**Cómo debe verse el resultado**
Un repositorio público que alguien puede clonar y reproducir, un dashboard con capturas en el README, una presentación que se entiende sin ti y una recomendación que cabe en un párrafo.

---
## 📣 Post 3: publica tu proyecto completo

Es la publicación de cierre: cuenta **qué recomendarías y por qué**, y enlaza el repositorio final.

**Estructura sugerida**
1. **Gancho** (1 o 2 líneas): tu mensaje principal, con su número.
2. **El recorrido**: de la pregunta a la recomendación en 3 o 4 líneas (datos, análisis, modelo o segmentos, dashboard).
3. **La recomendación** y su impacto estimado, con el supuesto principal.
4. **Qué aprendiste**: una decisión difícil y cómo la resolviste.
5. **Enlaces** al repositorio y, si lo publicaste, al dashboard, con una pregunta abierta.

**Imágenes**: la captura de la página de resumen del dashboard como portada y, si la plataforma lo permite, tu resumen ejecutivo o dos diapositivas clave. Míralas en el celular antes de publicar.

**Antes de publicar, revisa que:**
- [ ] cada cifra coincide con el dashboard, el notebook y el README;
- [ ] citas todas las fuentes con su licencia;
- [ ] no aparece ningún dato personal;
- [ ] el repositorio se puede reproducir siguiendo el README;
- [ ] las imágenes se leen en una pantalla pequeña y tienen texto alternativo;
- [ ] la recomendación dice sus supuestos y sus límites.

Escríbelo con tu voz: cuenta qué te sorprendió y qué harías distinto la próxima vez.

---
## ✅ Cierre: autoevaluación
Marca lo que puedes hacer sin mirar:
- [ ] Formatear cifras en soles, millones y porcentajes con funciones reutilizables.
- [ ] Armar una tabla ejecutiva ordenada, con total y sin índice.
- [ ] Escribir un mensaje principal que empiece por la conclusión y tenga un número.
- [ ] Estructurar una propuesta: contexto, hallazgos, recomendación, impacto, riesgos y siguiente paso.
- [ ] Preparar y exportar un gráfico con título-conclusión, énfasis y fuente.
- [ ] Citar fuentes con un formato consistente y revisar su licencia.
- [ ] Dejar un notebook que se ejecuta de principio a fin y se entiende sin mí.

---
## 🎓 Fin del plan
Terminaste las 27 sesiones: Python, NumPy, pandas, Matplotlib, Machine Learning, marketing analytics, aprendizaje no supervisado, SQL, series de tiempo y comunicación. Lo que queda es el cierre del proyecto (P4), el Post 3 y la presentación final. Marca tu avance en la sección **5. Progreso** de la currícula.